In [ ]:
import os
import sys

ROOT_PATH = '../'
os.chdir(ROOT_PATH)

sys.path.append(os.path.abspath(ROOT_PATH))

lib_path = os.path.abspath(os.path.join(ROOT_PATH, 'lib'))
if lib_path not in sys.path:
  sys.path.append(lib_path)

In [ ]:
from types import SimpleNamespace

import torch
from peft import LoraConfig, get_peft_model
from ultralytics.utils.loss import v8SegmentationLoss, E2ELoss
from torch.optim import AdamW
from torch.nn import L1Loss

from datasets.rescuenet_dataset import RescueNetDataset, collate_fn
from utils.augmentations import get_augmentation_pipeline
from model.TriheadSegmentationModel import TriheadSegmentationModel
from model.UncertaintyLossWeighting import UncertaintyLossWeighting
from utils.training import compute_normal_loss, EarlyStoppingAndCheckpointing
from custom_types.training import AblationStudyType
from utils.runpod import end_session

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

In [ ]:
MODEL_WEIGHT_DIR = 'model/weights'

In [ ]:
def create_peft_model(model: TriheadSegmentationModel):
  valid_target_modules = []
  for name, module in model.yolo.model.named_modules():
    if isinstance(module, torch.nn.Conv2d):
      if module.groups == 1:
        valid_target_modules.append(name)
  
  lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=valid_target_modules,
    modules_to_save=["model.23.*"],
    bias="none",
  )
  peft_model = get_peft_model(model.yolo.model, peft_config=lora_config).to(device)
  peft_model.print_trainable_parameters()
  peft_model.to(device)
  
  return peft_model

def infuse_args(model: TriheadSegmentationModel):
  current_args = model.yolo.model.args if isinstance(model.yolo.model.args, dict) else {}

  if 'overlap_mask' not in current_args:
    current_args['overlap_mask'] = True

  current_args['box'] = 7.5
  current_args['cls'] = 0.5
  current_args['dfl'] = 1.5
    
  model.yolo.model.args = SimpleNamespace(**current_args)

def get_model(model_type: AblationStudyType):
  if model_type == 'vanilla':
    model = TriheadSegmentationModel(
      yolo_pt_path=f'{MODEL_WEIGHT_DIR}/yolo26m-seg.pt',
      include_depth=False,
      include_normals=False,
      device=device
    )
  elif model_type == 'additional-depth':
    model = TriheadSegmentationModel(
      yolo_pt_path=f'{MODEL_WEIGHT_DIR}/yolo26m-seg.pt',
      include_depth=True,
      include_normals=False,
      device=device
    )
  elif model_type == 'additional-normal':
    model = TriheadSegmentationModel(
      yolo_pt_path=f'{MODEL_WEIGHT_DIR}/yolo26m-seg.pt',
      include_depth=False,
      include_normals=True,
      device=device
    )
  elif model_type == 'additional-both':
    model = TriheadSegmentationModel(
      yolo_pt_path=f'{MODEL_WEIGHT_DIR}/yolo26m-seg.pt',
      include_depth=True,
      include_normals=True,
      device=device
    )
  
  infuse_args(model)
  peft_model = create_peft_model(model)
  loss_balancer = UncertaintyLossWeighting()
  
  return model, peft_model, loss_balancer

In [ ]:
spatial_aug, photometric_aug = get_augmentation_pipeline()

def get_dataset_and_loader(model_type: AblationStudyType):
  if model_type == 'vanilla':
    include_depth = False
    include_normals = False
  elif model_type == 'additional-depth':
    include_depth = True
    include_normals = False
  elif model_type == 'additional-normal':
    include_depth = False
    include_normals = True
  elif model_type == 'additional-both':
    include_depth = True
    include_normals = True
  
  train_dataset = RescueNetDataset(data_dir='./data/RescueNet/train', include_depth=include_depth, spatial_transform=spatial_aug, photometric_transform=photometric_aug)
  val_dataset = RescueNetDataset(data_dir='./data/RescueNet/val', include_depth=include_depth, include_normals=include_normals)
  test_dataset = RescueNetDataset(data_dir='./data/RescueNet/test', include_depth=include_depth, include_normals=include_normals)
  
  train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=collate_fn)
  val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)
  test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)
  
  return {
    'train': (train_dataset, train_loader),
    'val': (val_dataset, val_loader),
    'test': (test_dataset, test_loader),
  }

In [ ]:
TOTAL_EPOCHS = 600

In [ ]:
for model_type in ['vanilla', 'additional-depth', 'additional-normal', 'additional-both']:
  original_model, peft_model, loss_balancer = get_model(model_type)
  dataset_and_loader = get_dataset_and_loader(model_type)
  
  trainable_params = [p for p in peft_model.parameters() if p.requires_grad]
  trainable_params.extend(loss_balancer.parameters())

  optimizer = AdamW(trainable_params, lr=1e-4, weight_decay=1e-4)

  seg_loss_criterion = E2ELoss(original_model.yolo.model, loss_fn=v8SegmentationLoss)
  depth_loss_criterion = L1Loss()
  
  early_stopping = EarlyStoppingAndCheckpointing()

  train_loader = dataset_and_loader['train'][1]
  val_loader = dataset_and_loader['val'][1]

  for epoch in range(TOTAL_EPOCHS):
    original_model.train()
    peft_model.train()
    
    for batch_idx, (batch_images, batch_targets) in enumerate(train_loader):
      optimizer.zero_grad()
      
      segmentation_out, depth_out, normal_out = peft_model(batch_images.to(device=device))
      
      depth_out = depth_out.to(device=device)
      normal_out = normal_out.to(device=device)
      
      true_segmentation_map = batch_targets[0]
      true_depth_map = batch_targets[1]['depth'].to(device=device)
      true_surface_normals = batch_targets[2]['normals'].to(device=device)

      seg_loss, seg_loss_items = seg_loss_criterion(segmentation_out, true_segmentation_map)
      depth_loss = depth_loss_criterion(depth_out, true_depth_map)
      normal_loss = compute_normal_loss(normal_out, true_surface_normals)
      
      loss_total = (torch.abs(loss_balancer.alpha).to(device) * seg_loss) + (torch.abs(loss_balancer.beta).to(device) * depth_loss) + (torch.abs(loss_balancer.gamma).to(device) * normal_loss)
      loss_total = loss_total.mean()
      loss_total.backward()
      
      optimizer.step()
      
      halt = False
      
    original_model.eval()
    peft_model.eval()
    with torch.no_grad():
      for batch_images_val, batch_targets_val in val_loader:
        segmentation_out, depth_out, normal_out = peft_model(batch_images_val.to(device=device))
    
        depth_out = depth_out.to(device=device)
        normal_out = normal_out.to(device=device)
        
        true_segmentation_map = batch_targets_val[0]
        true_depth_map = batch_targets_val[1]['depth'].to(device=device)
        true_surface_normals = batch_targets_val[2]['normals'].to(device=device)

        seg_loss, seg_loss_items = seg_loss_criterion(segmentation_out, true_segmentation_map)
        depth_loss = depth_loss_criterion(depth_out, true_depth_map)
        normal_loss = compute_normal_loss(normal_out, true_surface_normals)
    
        val_loss_total = (torch.abs(loss_balancer.alpha).to(device) * seg_loss) + (torch.abs(loss_balancer.beta).to(device) * depth_loss) + (torch.abs(loss_balancer.gamma).to(device) * normal_loss)
        val_loss_total = val_loss_total.mean()
        
        halt = early_stopping.record_and_check_if_halt(val_loss_total, peft_model.state_dict())
    
    print(
      f"Epoch [{epoch:03d}/{TOTAL_EPOCHS:03d}] Batch [{batch_idx:04d}/{len(train_loader):04d}] | "
      f"Total Loss: {loss_total.item():.4f} │ "
      f"Seg: {seg_loss.item():.4f} │ "
      f"Depth: {depth_loss.item():.4f} │ "
      f"Norm: {normal_loss.item():.4f} │ "
      f"LR: {optimizer.param_groups[0]['lr']:.2e}", end='\r'
    )
    
    if halt:
      break

In [ ]:
end_session()